# Notebook 01 - Problem Setup, Dataset, and Legal AI Foundations

This notebook is a zero-to-hero entry point: it explains the problem, the legal data, and why domain-specific fine-tuning is justified.

## Why Legal Summarization Is Difficult

1. Long-context dependencies can spread obligations and exceptions across distant sections.
2. Legal terminology has specialized meanings that differ from plain language.
3. Ambiguity and nested clauses can invert obligations based on exceptions.
4. Cross-references and jurisdiction-specific wording change enforceability.
5. Hallucinations can create legal risk by inventing obligations or liabilities.

## General-Purpose LLMs vs Domain-Specific Legal Fine-Tuning

General-purpose models can produce fluent summaries but often miss legal structure consistency. Domain-specific fine-tuning improves schema adherence, extraction reliability, and risk signaling for repeated legal workflows.

## What Is This Technique? - Dataset-Grounded Legal Task Framing

### Definition and Core Concepts
We frame the task as structured legal analysis grounded in real LexGLUE legal datasets, not synthetic prompt demos.

### Why Was This Technique Developed?
Prompt-only systems are easy to start but often inconsistent. A dataset-grounded setup is needed for objective baselines and measurable improvement.

### What Limitations of Traditional RAG Does It Solve?
Traditional RAG improves grounding at inference time but does not directly teach a model how to emit a stable legal-analysis schema. This project addresses output-structure learning via supervised fine-tuning.

### Architecture and Workflow Diagram Explanation
```mermaid
flowchart LR
A[LexGLUE datasets] --> B[Normalization + split integrity]
B --> C[Policy mapping to structured legal targets]
C --> D[Baselines]
C --> E[Fine-tuning]
D --> F[Evaluation + LLM judges]
E --> F
F --> G[Inference pipeline]
```

### Component-by-Component Breakdown
1. Dataset ingestion preserving train/validation/test boundaries.
2. Deterministic policy mapping from legal labels to structured outputs.
3. Baseline systems for prompt-only and zero-shot variants.
4. Fine-tuning stage with parameter-efficient adaptation.
5. Evaluation with metrics, judges, hallucination analysis, and latency.

### When Should It Be Used in Real-World Systems?
Use this setup when you need repeatable legal analysis outputs and auditable model progress.

### Advantages and Disadvantages
Advantages:
- Reproducible and measurable.
- Works fully local.
- Suitable as a reusable legal AI component.

Disadvantages:
- Requires data engineering effort.
- Gold labels are indirect for some target fields (risk/liability).
- Small models still have long-context limits.

### Comparison Against Standard RAG and Other Implemented Variants
Compared to standard RAG alone, this method teaches output structure through training. Compared to prompt-only baselines, it is more expensive initially but more consistent.

### Implementation Details and Design Decisions Used in This Project
We use LexGLUE subsets: LEDGAR, UNFAIR-ToS, SCOTUS, EURLEX, CaseHOLD. Targets are generated deterministically with explicit policy rules to remain auditable.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
ROOT = next((p for p in candidates if (p / 'scripts').exists() and (p / 'configs').exists()), cwd)
print('Project root:', ROOT)

def run_py(script: str, *args: str) -> None:
    cmd = [sys.executable, str(ROOT / script), *args]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=str(ROOT))


In [ ]:
report_path = ROOT / 'data/processed/dataset_report.json'
if not report_path.exists():
    run_py('scripts/prepare_data.py', '--config', 'configs/default.yaml')
report = json.loads(report_path.read_text())
report

## Dataset Structure and Labeling Schema

- Mix of single-label and multi-label legal tasks.
- Metadata includes dataset identity, split, and legal label names.
- Regulatory hierarchy represented through mixed domains: contract law, US case law, EU law.

In [ ]:
fig_path = ROOT / 'artifacts/figures/dataset_distributions.png'
img = plt.imread(fig_path)
plt.figure(figsize=(12, 4))
plt.imshow(img)
plt.axis('off')
plt.title('Dataset and Risk Distributions')
plt.show()

## Dataset Limitations

1. No native executive-summary labels across all subsets.
2. No direct gold risk/liability taxonomy for our final schema.
3. Cross-jurisdiction semantic differences remain challenging for compact models.

These limitations are handled with deterministic policy mapping and explicit evaluation.